# 02 - Baselines
Referências simples, avaliadas na mesma partição temporal e com as mesmas métricas do modelo.

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from src.config import *
from src.dados import *
from src.variaveis import *
np.random.seed(SEMENTE)
PROCESSADO.mkdir(parents=True, exist_ok=True)
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
from src.particao import particionar_temporal
from src.baseline import *

In [2]:
bruto = carregar_bruto()
assert bruto is not None, "Coloque a planilha em dados/bruto/ (veja dados/bruto/LEIAME.md)"
bruto = bruto.drop(columns="arquivo_origem")
base, log, total_entrada = preparar_base_mensal(bruto)
feats, sem_alvo = criar_variaveis(base)
treino, valid, teste = particionar_temporal(feats)
for nome, d in [("treino",treino),("validação",valid),("teste",teste)]:
    print(f"{nome:10s} {d[COL_MES].min():%Y-%m} a {d[COL_MES].max():%Y-%m}  | {d[COL_MES].nunique()} meses | {len(d)} linhas")

treino     2025-08 a 2026-03  | 8 meses | 48 linhas
validação  2026-04 a 2026-06  | 3 meses | 18 linhas
teste      2026-07 a 2026-09  | 3 meses | 18 linhas


## Baselines na validação
- **Trivial**: média histórica de cada origem.
- **Operacional (hipótese da equipe)**: repetir o último mês observado de cada origem.

In [3]:
prev = {"trivial: média por origem": prever_media_treino(treino, valid),
        "trivial: mediana por origem": prever_media_treino(treino, valid, "median"),
        "operacional: último valor": prever_ultimo_valor(treino, valid)}
tab = tabela_metricas(valid, prev).round(3); display(tab)
tab.to_csv(PROCESSADO/"baselines_validacao.csv", index=False)

,abordagem,MAE do total mensal (principal),WAPE por série (secundária),viés do total mensal
0,trivial: média por origem,140.583,0.169,16.917
1,trivial: mediana por origem,146.000,0.161,54.667
2,operacional: último valor,263.667,0.272,263.667
